# 02 — Build the aligned multimodal artifact

This notebook turns the feature/label rows into an aligned multimodal `.npz`.

It builds:

- `tabular_tokens`
- `image_tokens`
- `text_tokens`
- `kg_tokens`
- `y`
- `stock_ids`
- `end_dates`

This is the main bridge from raw multimodal data to embedding space.

In [ ]:
import os
REPO_DIR = "/content/nifty50-multimodal-transformer"
if os.path.exists(REPO_DIR):
    os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

from src.data.download_yfinance import deterministic_csv_path_for_ticker
from src.data.multimodal_samples import (
    attach_image_tokens,
    attach_kg_tokens,
    attach_text_tokens,
    build_tabular_multimodal_samples,
    save_multimodal_samples,
)
from src.kg.build_graph import build_market_knowledge_graph
from src.models.image_transformer import ImageTransformerConfig
from src.viz.charts import generate_or_resolve_sample_chart

TICKERS = ["RELIANCE.NS", "TCS.NS", "INFY.NS"]
BENCHMARK = "^NSEI"
SECTORS = {"RELIANCE.NS": "Energy", "TCS.NS": "IT", "INFY.NS": "IT"}

OUTPUT_DIR = Path("data/processed/real_world_demo")
RAW_DIR = OUTPUT_DIR / "raw"
CHART_DIR = OUTPUT_DIR / "charts"
WINDOW_SIZE = 20
CHART_LOOKBACK_DAYS = 20
TEXT_DIM = 16

FEATURE_COLUMNS = [
    "log_return_1d",
    "cum_return_3d",
    "cum_return_5d",
    "cum_return_10d",
    "realized_vol_5d",
    "realized_vol_10d",
    "high_low_range_over_close",
    "close_over_10dma_minus_1",
    "close_over_20dma_minus_1",
    "volume_over_20d_avg",
    "stock_minus_index_return",
]

In [ ]:
tabular_df = pd.read_csv(OUTPUT_DIR / "tabular_samples.csv", parse_dates=["date"])

arrays = build_tabular_multimodal_samples(
    tabular_df,
    feature_cols=FEATURE_COLUMNS,
    window_size=WINDOW_SIZE,
)
print("Tabular tokens:", arrays.tabular_tokens.shape)
print("Samples:", len(arrays.y))

In [ ]:
# Create leakage-safe text records derived from real market rows.
records = []
for ticker, frame in tabular_df.groupby("stock_id"):
    frame = frame.sort_values("date").reset_index(drop=True)
    for idx in range(0, len(frame), 5):
        row = frame.iloc[idx]
        direction = "positive" if row["log_return_1d"] >= 0 else "negative"
        records.append({
            "stock_id": ticker,
            "event_date": row["date"],
            "source_type": "market_summary",
            "title": f"{ticker} {direction} daily market summary",
            "body_text": (
                f"As of {row['date'].date()}, {ticker} had a {direction} one-day return "
                f"of {row['log_return_1d']:.4f}, relative return versus index "
                f"of {row['stock_minus_index_return']:.4f}, and volume ratio "
                f"{row['volume_over_20d_avg']:.4f}."
            ),
        })

text_records = pd.DataFrame(records)
text_records_csv = OUTPUT_DIR / "text_records.csv"
text_records.to_csv(text_records_csv, index=False)
display(text_records.head())

In [ ]:
# Build KG context inputs.
stock_sectors = pd.DataFrame([{"stock_id": t, "sector_id": SECTORS.get(t, "Unknown")} for t in TICKERS])
stock_sectors.to_csv(OUTPUT_DIR / "stock_sectors.csv", index=False)

kg_returns = tabular_df.loc[:, ["stock_id", "date", "stock_minus_index_return"]].rename(
    columns={"stock_minus_index_return": "recent_return"}
)
kg_returns.to_csv(OUTPUT_DIR / "kg_returns.csv", index=False)

events = []
for ticker, frame in tabular_df.groupby("stock_id"):
    top_volume = frame["volume_over_20d_avg"].quantile(0.9)
    for _, row in frame.loc[frame["volume_over_20d_avg"] >= top_volume].iterrows():
        events.append({"stock_id": ticker, "event_date": row["date"], "event_type": "high_volume"})
event_records = pd.DataFrame(events) if events else pd.DataFrame(columns=["stock_id", "event_date", "event_type"])
event_records.to_csv(OUTPUT_DIR / "event_records.csv", index=False)

display(stock_sectors)
display(event_records.head())

In [ ]:
# Generate candlestick chart PNGs for each aligned sample.
stock_data = {
    ticker: pd.read_csv(deterministic_csv_path_for_ticker(ticker, RAW_DIR), parse_dates=["date"])
    for ticker in TICKERS
}

for stock_id, end_date in zip(arrays.stock_ids, arrays.end_dates):
    generate_or_resolve_sample_chart(
        stock_data[str(stock_id)],
        symbol=str(stock_id),
        prediction_date=pd.Timestamp(end_date),
        output_dir=CHART_DIR,
        lookback_days=CHART_LOOKBACK_DAYS,
    )

print("Charts generated:", len(list(CHART_DIR.glob("*.png"))))

In [ ]:
# Attach text, KG, and image tokens.
graph = build_market_knowledge_graph(SECTORS, event_records=event_records)

arrays = attach_text_tokens(arrays, text_records, dim=TEXT_DIM)
arrays = attach_kg_tokens(arrays, graph, returns=kg_returns)
arrays = attach_image_tokens(
    arrays,
    chart_dir=CHART_DIR,
    config=ImageTransformerConfig(
        image_size=64,
        patch_size=16,
        model_dim=16,
        num_heads=4,
        num_layers=1,
        ff_dim=32,
    ),
    device="cpu",
)

dataset_path = save_multimodal_samples(arrays, OUTPUT_DIR / "real_world_multimodal_samples.npz")
print("Wrote:", dataset_path)

In [ ]:
# Inspect actual tensor shapes.
data = np.load(OUTPUT_DIR / "real_world_multimodal_samples.npz", allow_pickle=False)
for key in data.files:
    print(f"{key:16s}", data[key].shape, data[key].dtype)

In [ ]:
# Show a few real generated candlestick charts.
from IPython.display import Image, display
for path in list(CHART_DIR.glob("*.png"))[:3]:
    print(path.name)
    display(Image(filename=str(path)))

## Checkpoint output

This notebook writes:

```text
data/processed/real_world_demo/real_world_multimodal_samples.npz
data/processed/real_world_demo/charts/*.png
data/processed/real_world_demo/text_records.csv
data/processed/real_world_demo/kg_returns.csv
data/processed/real_world_demo/event_records.csv
```

Next notebook: `03_train_fusion_and_ablate.ipynb`.